In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from polytensor import PolyTensor

torch.manual_seed(0)
device = torch.device('cpu')
deg = 2

# Tiny input + first-order perturbation (higher coefficients start at zero).
x0 = torch.randn(2, 3, 8, 8, device=device)
x1 = 0.01 * torch.randn_like(x0)
x = PolyTensor.from_primal(x0, degree=deg, tangent=x1, requires_grad=False)

# Tiny model pieces.
conv = nn.Conv2d(3, 4, kernel_size=3, padding=1, bias=True).to(device)
bn = nn.BatchNorm2d(4, eps=1e-5, momentum=0.1).to(device)
pool = nn.MaxPool2d(2)
head = nn.Linear(4 * 4 * 4, 5).to(device)

# Forward path using requested ops.
h = conv(x)
h = bn(h)
h = nn.GELU()(h)
h = pool(h)
h = h.view(h.shape[0], -1)
logits = head(h)

targets = torch.tensor([1, 3], device=device)
ce = F.cross_entropy(logits, targets)
lse = torch.logsumexp(logits, dim=1).mean()
nm = logits.norm(dim=1).mean()
loss = ce + 0.01 * lse + 0.01 * nm

print('logits coeffs shape:', tuple(logits.coeffs.shape))
print('ce coeffs:', ce.coeffs)
print('norm coeffs:', nm.coeffs)
print('total loss coeffs:', loss.coeffs)

/Users/gunn/projects/data-deletion-code/poly-tensors/.venv/lib/python3.12/site-packages/torch/nn/modules/transformer.py:20: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  device: torch.device = torch.device(torch._C._get_default_device()),  # torch.device('cpu'),


KeyboardInterrupt: 

In [ ]:
# Backward through polynomial coefficients into real-valued parameters.
for p in list(conv.parameters()) + list(bn.parameters()) + list(head.parameters()):
    if p.grad is not None:
        p.grad = None

loss.backward()

print('||grad conv.weight||:', float(conv.weight.grad.norm()))
print('||grad head.weight||:', float(head.weight.grad.norm()))